# Finding most popular locations mentioned in TikTok Data
Method:
- create df with just post_id, description_cleaned
- input description to LLM which will output structure JSON of locations 
- count in pandas
- edit locations 

In [342]:
import json
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

### Loading csv file

In [343]:
folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/com~apple~CloudDocs"
    "/AC Work/Navigator/navigator-travel-trends-intelligence/data/tiktok/processed/"
)

combined_data = pd.read_csv(folder / f"combined_clean.csv")

# combined_data.head()
combined_data.shape

(3725, 28)

### Turn into df with only post_id, destination, description_clean

In [344]:
columns = [
    "post_id",
    "destination",
    "description_clean"
]

description_df = combined_data[columns]

print(description_df.shape)
# print(description_df.head())
print(combined_data.head)
print(type(description_df["post_id"]))
# print(description_df.tail())

(3725, 3)
<bound method NDFrame.head of                                                     url              post_id  \
0     https://www.tiktok.com/@alinabadrudin/video/76...  7631556131969666326   
1     https://www.tiktok.com/@suitelifeoflauren/vide...  7639377582923844894   
2     https://www.tiktok.com/@heeral.kakkad/video/76...  7647610856905133333   
3     https://www.tiktok.com/@cesca.dia/video/763651...  7636514805171490061   
4     https://www.tiktok.com/@babygirls.in.portugal/...  7657857706417556766   
...                                                 ...                  ...   
3720  https://www.tiktok.com/@aliciagigs/video/72259...  7225951190457011474   
3721  https://www.tiktok.com/@gabesco/video/76472889...  7647288952457661710   
3722  https://www.tiktok.com/@travel_hayley/video/71...  7179702432476171525   
3723  https://www.tiktok.com/@andreachen88/video/763...  7639098761951579406   
3724  https://www.tiktok.com/@mammajonoexcusetravel/...  7486506444330634518   


### (Using NER to first locate places)

In [345]:
# nlp = spacy.load("en_core_web_sm")

# locations = []

# for text in description_df["description"].fillna(""):
#     doc = nlp(text)

#     for entity in doc.ents:
#         if entity.label_ in ["LOC", "FAC"]:
#             locations.append(entity.text)

### Turning df into a csv to be inputted into a LLM

In [346]:
description_df.to_csv(
        "data/tiktok/processed/descriptions_clean.csv",
        index=False
    )

### Editing LLM JSON output with correct post_ids because some reason LLM changed the post_ids but maintained the order

In [347]:
locations_df = pd.read_json("data/tiktok/raw/tiktoks_locs.json")

locations_dict = locations_df.to_dict(orient="records")

print(len(combined_data))
print(len(locations_dict))

j = 0

print("actual", combined_data.iloc[j]["post_id"])
print(combined_data.iloc[j]["description_clean"])



print("old",locations_dict[j]["post_id"])
print("old",locations_dict[j]["locations"])

i = 0

for post in locations_dict:
    post_id = str(combined_data.iloc[i]["post_id"])
    post["post_id"] = post_id
    i += 1


print("new", locations_dict[j]["post_id"])



3725
3725
actual 7631556131969666326
i fear nando’s will never taste the same 😭 lisbon solotravel  experience a solo traveler's first lunch in lisbon, portugal, as they share their poignant first nando's experience. will it live up to expectations? lisbon solotravel embarking on a solo adventure in lisbon, portugal, this traveler shares a deeply personal moment: their first solo lunch in the country. the chosen eatery, nando's, holds a special significance, prompting an emotional reflection with a poignant "i fear nando’s will never taste the same 😭" the accompanying sticker text "first solo lunch in portugal 🇵🇹" further emphasizes the milestone of this experience. this video captures the essence of solo travel, from the excitement of new culinary discoveries to the introspective moments that come with navigating a foreign land alone.
old 7631556131969666048
old [{'name': "Nando's", 'type': 'restaurant', 'classifiers': ['food_drink']}]
new 7631556131969666326


### Editing LLM JSON output to ignore locations with type = food and replacing locs that repeat (Madrasa Ben Youssef) and ones with different names to the google maps reviews locations

In [348]:

for post in locations_dict:

    # replacing locations
    for location in post["locations"]:
        
        if location["name"] == "Ben Youssef Madrasa":
            location["name"] = "Madrasa Ben Youssef"
        
        if location["name"] == "Time Out Market Lisboa":
            location["name"] = "Time Out Market"
        
        if location["name"] == "MAAT":
            location["name"] = "MAAT - Museum of Art, Architecture and Technology"
        
        if location["name"] == "Carmo Convent":
            location["name"] = "Carmo Archaeological Museum"

        if location["name"] == "Tuk Tuk Lisboa":
            location["name"] = "Tuk Tuk Lisbon Tours"

        if location["name"] == "Besa Restaurante":
            location["name"] = "Bessa Restaurante(seafood restaurant ))"

        if location["name"] == "Machi Machimbombo":
            location["name"] = "Machimbombo"

        if location["name"] == "Praia da Ursa":
            location["name"] = "Ursa Beach"

        if location["name"] == "Pena Palace":
            location["name"] = "National Palace of Pena"
        
        if location["name"] == "Souks of Marrakech":
            location["name"] = "Souk Semmarine"
        
        if location["name"] == "Koutoubia Mosque":
            location["name"] = "Koutoubia"
        
        if location["name"] == "Bacha Coffee Marrakech":
            location["name"] = "Bacha Coffee"

        if location["name"] == "Palmeraie":
            location["name"] = "Palmeraie Marrakech chameau"

        if location["name"] == "Hanoi Train Street":
            location["name"] = "Train Street"

        if location["name"] == "Ho Chi Minh Mausoleum":
            location["name"] = "Ho Chi Minh's Mausoleum"

        if location["name"] == "St. Joseph’s Cathedral":
            location["name"] = "St Joseph Cathedral"
        
        if location["name"] == "Hanoi Night Market":
            location["name"] = "Chợ Hàng Mã"

        if location["name"] == "Imperial Citadel of Thăng Long":
            location["name"] = "Imperial Citadel of Thang Long"

        if location["name"] == "Café Giảng":
            location["name"] = "Cafe Giảng"

        if location["name"] == "Bánh Mì 25":
            location["name"] = "Banh Mi 25"

        if location["name"] == "Bún Chả Hương Liên":
            location["name"] = "Bún chả Hương Liên"

        if location["name"] == "Lotte Observation Deck":
            location["name"] = "Sky Lotte Observation Deck"

        if location["name"] == "Temple of Literature":
            location["name"] = "Temple Of Literature"
        
        if location["name"] == "Reykjavík Old Harbour":
            location["name"] = "Harbor in Reykjavik"
        
        if location["name"] == "Hallgrímskirkja":
            location["name"] = "Hallgrimskirkja"

        if location["name"] == "Jökulsárlón Glacier Lagoon":
            location["name"] = "Jökulsárlón"

        if location["name"] == "Harpa Concert Hall":
            location["name"] = "Harpa Concert Hall and Conference Centre"

        if location["name"] == "Icelandic Phallological Museum":
            location["name"] = "The Icelandic Phallological Museum"

        if location["name"] == "Gullfoss":
            location["name"] = "Gullfoss Falls"

        if location["name"] == "Rainbow Street":
            location["name"] = "Skólavörðustígur Rainbow Street"

    # removing is location type = food and if name = alfama (no google maps reviews)
    post["locations"] = [
    location
    for location in post["locations"]
    if (
        location["type"] != "food"
        and location["name"] != "Alfama"
        and location["name"] != "Bairro Alto"
        and location["name"] != "Cascais"
        and location["name"] != "Laugavegur"
    )
]

# check - false if all entries with type = food has been removed and Madrasa Ben Youssef has been replaced
print(
    any(
        location.get("type") == "food" 
        and location.get("name") == "Madrasa Ben Youssef"
        for locations in locations_df["locations"]
        for location in locations
    )
)

print(type(locations_dict[0]["post_id"]))
print(locations_dict[0]["post_id"])

False
<class 'str'>
7631556131969666326


### Checking for duplicate locations in each post and removing them

In [349]:
# checking for duplicates
for post in locations_dict:
    location_names = [
        location["name"]
        for location in post["locations"]
    ]

    if len(location_names) != len(set(location_names)):
        print(
            f"Duplicate location in post: "
            f"{post['post_id']}"
        )

# removing duplicates
for post in locations_dict:
    seen = set()
    unique_locations = []

    for location in post["locations"]:
        if location["name"] not in seen:
            unique_locations.append(location)
            seen.add(location["name"])

    post["locations"] = unique_locations

# checking if removal of duplicates worked - should output False
duplicates = False

for post in locations_dict:
    location_names = [
        location["name"]
        for location in post["locations"]
    ]

    if len(location_names) != len(set(location_names)):
        duplicates = True

print(duplicates)

Duplicate location in post: 7664525661549104406
Duplicate location in post: 7515180352453823766
Duplicate location in post: 7460656174157958431
Duplicate location in post: 7528702154916252950
Duplicate location in post: 7625765121167772931
Duplicate location in post: 7539547397064543519
Duplicate location in post: 7387460395541237024
False


### Counting locations mentioned 

In [350]:
location_counts = defaultdict(Counter)

for post in locations_dict:
    destination = post["destination"]

    for location in post["locations"]:
        location_name = location["name"]
        location_counts[destination][location_name] += 1

print(location_counts)

defaultdict(<class 'collections.Counter'>, {'lisbon': Counter({'Belém Tower': 55, 'Chiado': 45, 'LX Factory': 43, 'Jerónimos Monastery': 39, 'Time Out Market': 36, 'Pastéis de Belém': 35, 'Praça do Comércio': 35, 'Pink Street': 32, 'National Palace of Pena': 25, 'Miradouro da Senhora do Monte': 20, 'Santa Justa Lift': 18, 'MAAT - Museum of Art, Architecture and Technology': 10, 'Miradouro de São Pedro de Alcântara': 9, 'Castelo de São Jorge': 8, 'Miradouro de Santa Catarina': 7, 'Carmo Archaeological Museum': 7, 'Oceanário de Lisboa': 5, 'Tuk Tuk Lisbon Tours': 3, 'Bar Alimentar': 2, "Nando's": 1, 'Bessa Restaurante(seafood restaurant ))': 1, 'Machimbombo': 1, 'Ursa Beach': 1}), 'marrakech': Counter({'Souk Semmarine': 132, 'Medina of Marrakech': 113, 'Jardin Majorelle': 60, 'Madrasa Ben Youssef': 43, 'Agafay Desert': 38, 'Bahia Palace': 29, 'Atlas Mountains': 28, 'Jemaa el-Fnaa': 27, 'Yves Saint Laurent Museum': 25, 'Koutoubia': 23, 'Bacha Coffee': 22, 'El Badi Palace': 16, 'Le Jardin 

### Saving to JSON to view and check

In [351]:
with open("data/tiktok/raw/tiktoks_locs_counts.json", "w") as f:
    json.dump(
        location_counts,
        f,
        indent=2,
        ensure_ascii=False
    )

### Editing loc counts, removing invalid locations (not at the destination or not a specific location on Google Maps) and sorting

In [352]:
deleted_locs = ["Nando's", "Medina of Marrakech", "Atlas Mountains", "Sapa", "Ha Long Bay", "Ninh Bình", "Da Nang", "Northern Lights", "Golden Circle", "South Coast"]


for locations in location_counts.values():

    for deleted_loc in deleted_locs:
        locations.pop(deleted_loc, None)

# checking if desired locs are deleted - should return false
still_present = False

for deleted_loc in deleted_locs:

    for destination, locations in location_counts.items():

        if deleted_loc in locations:
            still_present = True

print(still_present)

# sorting locations_counts
location_counts_sorted = {
    destination: dict(
        sorted(
            locations.items(),
            key=lambda item: item[1],
            reverse=True
        )
    )
    for destination, locations in location_counts.items()
}

False


### Making new file tiktoks_locs_count_clean.json 

In [353]:
with open("data/tiktok/processed/tiktoks_locs_count_clean.json", "w") as f:
    json.dump(location_counts_sorted, f, indent=2, ensure_ascii=False)

### Removing deleted loccs from locations_dict

In [354]:
# removing deleted locs from tiktoks_locs
for post in locations_dict:

    post["locations"] = [
        location
        for location in post["locations"]
        if location["name"] not in deleted_locs
    ]

# checks - should both return false, false, true
print(
    any(
        location["name"] in deleted_locs
        for post in locations_dict
        for location in post["locations"]
    )
)

print(
    any(
        location["name"] == "Madrasa Ben Youssef"
        for post in locations_dict
        for location in post["locations"]
    )
)
print(
    any(
        location["name"] == "Ben Youssef Madrasa"
        for post in locations_dict
        for location in post["locations"]
    )
)


False
True
False


### Saving new file tiktoks_locs_clean.json

In [355]:
with open("data/tiktok/processed/tiktoks_locs_clean.json", "w") as f:
    json.dump(locations_dict, f, indent=2, ensure_ascii=False)